# 01 - Merge and prep
This file merges all csv files per crisis and adds a label of what phase the comment was made

Merges the five phase result files into one table (`analysis_df.csv`) for `02-statistical-analysis`.
Run from the repo root. Check the paths and `OK_STATUS` in the config cell first.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd


ARCHIVE = Path("03-llm-passes/archive/2026-10-05_v1.4_4bit")
CRISIS = "ateneo-drowning"
PHASES = ["01-onset", "02-confirmation", "03-spread", "04-reckoning", "05-lingering"]
OK_STATUS = None
OUT = Path("04-statistical-analysis/output") / CRISIS
OUT.mkdir(parents=True, exist_ok=True)

## Load

In [ ]:
from pathlib import Path
base = ARCHIVE / CRISIS / "full-codebook-coding"
frames = []
for p in PHASES:
    f = base / p / "full_codebook_results.csv"
    print(p, f.exists())
    d = pd.read_csv(f, keep_default_na=False, na_values=[""])
    frames.append(d.assign(phase=p, source_file=f.name))
df = pd.concat(frames, ignore_index=True)
df.shape

## Checks before filtering

In [ ]:
print(df.groupby("phase").size(), "\n")
for c in ["status", "autofix", "confidence", "LANGUAGE"]:
    print(df[c].value_counts(dropna=False), "\n")
print("duplicate comment_id:", df.duplicated("comment_id").sum())

## Filter

In [ ]:
if OK_STATUS is not None:
    df = df[df.status == OK_STATUS]
df = df[pd.to_numeric(df.UNCODABLE, errors="coerce").fillna(0) == 0].copy()
len(df)

## Types and derived variables

In [ ]:
binary = ["ANGER", "CONTEMPT", "DISGUST", "SADNESS", "CONDOLENCE", "COMPASSION_VICTIMS",
          "SYMPATHY_OFFENDER", "FEAR", "NORM_VIOLATION", "IGNORED_WARNING", "RECURRENCE",
          "COVER_UP", "ALT_ACCOUNT", "INJUSTICE", "SELF_INTEREST", "NEEDS_CONTEXT",
          "CLASS_FRAME", "APPEARANCE_ATTACK", "META_DISCOURSE", "SARCASM"]
cats = ["ATTRIBUTION", "STANCE", "TARGET_OF_COMMENT", "CLASS_DIRECTION", "RESPONSE_TARGET",
        "RESPONSE_EVAL", "LANGUAGE", "confidence"]

df[binary] = df[binary].apply(pd.to_numeric, errors="coerce").fillna(0).astype(int)
for c in cats:
    df[c] = df[c].fillna("NA").astype(str).str.strip()

df["phase"] = pd.Categorical(df.phase, PHASES, ordered=True)
df["phase_num"] = df.phase.cat.codes

emotions = ["ANGER", "CONTEMPT", "DISGUST", "SADNESS", "CONDOLENCE",
            "COMPASSION_VICTIMS", "SYMPATHY_OFFENDER", "FEAR"]
df["NEG_EMOTION"] = df[["ANGER", "CONTEMPT", "DISGUST"]].max(axis=1)
df["ANY_EMOTION"] = df[emotions].max(axis=1)
df["log_len"] = np.log1p(df.text.str.len())

## Split multi-label columns

In [ ]:
multi = ["BLAME_TARGETS", "VALUES", "ACTIONS", "IDENTITY_FRAME", "REJECT_REASONS"]

def parse(x):
    if pd.isna(x):
        return []
    s = re.sub(r"[\[\]'\"]", "", str(x))
    return [t.strip() for t in re.split(r"[|;,]", s) if t.strip()]

for c in multi:
    tokens = df[c].apply(parse)
    for v in sorted({t for ts in tokens for t in ts} - {"NA"}):
        df[f"{c}__{v}"] = tokens.apply(lambda ts: int(v in ts))

[c for c in df.columns if "__" in c]

## Broad outrage

In [ ]:
blame = [c for c in df if c.startswith("BLAME_TARGETS__") and not c.endswith("__NONE")]
acts = [c for c in ["ACTIONS__PUNITIVE", "ACTIONS__RETALIATION"] if c in df]
targeted = (df[blame].max(axis=1).eq(1) | df.ATTRIBUTION.eq("ACCUSATION")
            | df[acts].max(axis=1).eq(1))
df["BROAD_OUTRAGE"] = (df.NEG_EMOTION.eq(1) & targeted).astype(int)

In [ ]:
no_dir = df.CLASS_DIRECTION.eq("NA")
resp_t, resp_e = df.RESPONSE_TARGET.eq("NA"), df.RESPONSE_EVAL.eq("NA")
rules = {
    "direction without class frame": df.CLASS_FRAME.eq(0) & ~no_dir,
    "class frame without direction": df.CLASS_FRAME.eq(1) & no_dir,
    "response target/eval mismatch": resp_t != resp_e,
    "needs_context but stance not neutral": df.NEEDS_CONTEXT.eq(1) & df.STANCE.ne("NEUTRAL"),
}
for k, m in rules.items():
    print(k, int(m.sum()))
df["flag_inconsistent"] = pd.concat(rules.values(), axis=1).any(axis=1).astype(int)

In [ ]:
print(pd.crosstab(df.phase, df.CLASS_FRAME, margins=True), "\n")
print(pd.crosstab(df.phase, df.CLASS_DIRECTION), "\n")
print(pd.crosstab(df.CLASS_FRAME, df.NEG_EMOTION, margins=True))

## Save

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT / "analysis_df.csv", index=False)
print(OUT.resolve() / "analysis_df.csv", df.shape)